In [1]:
import pandas as pd

df = pd.read_csv('BlinkIT Grocery Data.csv')

In [2]:
df.head()

,Item Fat Content,Item Identifier,Item Type,Outlet Establishment Year,Outlet Identifier,Outlet Location Type,Outlet Size,Outlet Type,Item Visibility,Item Weight,Total Sales,Rating
0,Regular,FDX32,Fruits and Vegetables,2012,OUT049,Tier 1,Medium,Supermarket Type1,0.100014,15.10,145.4786,5.0
1,Low Fat,NCB42,Health and Hygiene,2022,OUT018,Tier 3,Medium,Supermarket Type2,0.008596,11.80,115.3492,5.0
2,Regular,FDR28,Frozen Foods,2010,OUT046,Tier 1,Small,Supermarket Type1,0.025896,13.85,165.0210,5.0
3,Regular,FDL50,Canned,2000,OUT013,Tier 3,High,Supermarket Type1,0.042278,12.15,126.5046,5.0
4,Low Fat,DRI25,Soft Drinks,2015,OUT045,Tier 2,Small,Supermarket Type1,0.033970,19.60,55.1614,5.0


In [3]:
print(df.info)

<bound method DataFrame.info of      Item Fat Content Item Identifier              Item Type  \
0             Regular           FDX32  Fruits and Vegetables   
1             Low Fat           NCB42     Health and Hygiene   
2             Regular           FDR28           Frozen Foods   
3             Regular           FDL50                 Canned   
4             Low Fat           DRI25            Soft Drinks   
...               ...             ...                    ...   
8518          low fat           NCT53     Health and Hygiene   
8519          low fat           FDN09            Snack Foods   
8520          low fat           DRE13            Soft Drinks   
8521              reg           FDT50                  Dairy   
8522              reg           FDM58            Snack Foods   

      Outlet Establishment Year Outlet Identifier Outlet Location Type  \
0                          2012            OUT049               Tier 1   
1                          2022            OUT018  

In [4]:
df.isnull().sum()

Item Fat Content                0
Item Identifier                 0
Item Type                       0
Outlet Establishment Year       0
Outlet Identifier               0
Outlet Location Type            0
Outlet Size                     0
Outlet Type                     0
Item Visibility                 0
Item Weight                  1463
Total Sales                     0
Rating                          0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df['Item Fat Content'].value_counts()

Item Fat Content
Low Fat    5089
Regular    2889
LF          316
reg         117
low fat     112
Name: count, dtype: int64

In [ ]:
df['Item Fat Content'] = df['Item Fat Content'].replace(
    {'LF':'Low Fat',
    'low fat':'Low Fat',
    'reg':'Regular'})

In [14]:
df['Item Fat Content'].value_counts()

Item Fat Content
Low Fat    5517
Regular    3006
Name: count, dtype: int64

In [15]:
df["Item Weight"].isnull().sum()

np.int64(1463)

In [17]:
df["Item Weight"].value_counts()

Item Weight
12.150    86
17.600    82
13.650    77
11.800    76
15.100    68
          ..
8.920      2
5.400      1
7.685      1
6.520      1
9.420      1
Name: count, Length: 415, dtype: int64

In [18]:
df.groupby("Item Type")["Item Weight"].apply(lambda x: x.isnull().sum())

Item Type
Baking Goods             112
Breads                    47
Breakfast                 21
Canned                   110
Dairy                    116
Frozen Foods             138
Fruits and Vegetables    213
Hard Drinks               31
Health and Hygiene        90
Household                151
Meat                      88
Others                    32
Seafood                   13
Snack Foods              212
Soft Drinks               71
Starchy Foods             18
Name: Item Weight, dtype: int64

In [19]:
# Fill missing weights using the median of each specific Item Type
df["Item Weight"] = df["Item Weight"].fillna(
    df.groupby("Item Type")["Item Weight"].transform("median")
)


In [20]:
df.isnull().sum()

Item Fat Content             0
Item Identifier              0
Item Type                    0
Outlet Establishment Year    0
Outlet Identifier            0
Outlet Location Type         0
Outlet Size                  0
Outlet Type                  0
Item Visibility              0
Item Weight                  0
Total Sales                  0
Rating                       0
dtype: int64

In [22]:
df.describe()

,Outlet Establishment Year,Item Visibility,Item Weight,Total Sales,Rating
count,8523.000000,8523.000000,8523.000000,8523.000000,8523.000000
mean,2010.831867,0.066132,12.813390,140.992782,3.965857
std,8.371760,0.051598,4.241384,62.275067,0.605651
min,1998.000000,0.000000,4.555000,31.290000,1.000000
25%,2000.000000,0.026989,9.310000,93.826500,4.000000
50%,2012.000000,0.053931,12.850000,143.012800,4.000000
75%,2017.000000,0.094585,16.000000,185.643700,4.200000
max,2022.000000,0.328391,21.350000,266.888400,5.000000


## Feature Engineering


In [25]:
CURRENT_YEAR = 2026

df["Outlet Age"] = CURRENT_YEAR - df["Outlet Establishment Year"]

In [26]:
df["Sales Category"] = pd.qcut(
    df["Total Sales"],
    q=3,
    labels=["Low","Medium","High"]
)

In [27]:
df["Visibility Category"] = pd.qcut(
    df["Item Visibility"],
    q=3,
    labels=["Low","Medium","High"]
)

In [28]:
df["Rating Category"] = pd.cut(
    df["Rating"],
    bins=[0,3,4,5],
    labels=["Poor","Average","Excellent"]
)

In [29]:
df.columns

Index(['Item Fat Content', 'Item Identifier', 'Item Type',
       'Outlet Establishment Year', 'Outlet Identifier',
       'Outlet Location Type', 'Outlet Size', 'Outlet Type', 'Item Visibility',
       'Item Weight', 'Total Sales', 'Rating', 'Outlet Age', 'Sales Category',
       'Visibility Category', 'Rating Category'],
      dtype='str')

In [30]:
df.columns = (df.columns.str.strip().str.replace(" ", "_").str.lower())

In [31]:
df.columns

Index(['item_fat_content', 'item_identifier', 'item_type',
       'outlet_establishment_year', 'outlet_identifier',
       'outlet_location_type', 'outlet_size', 'outlet_type', 'item_visibility',
       'item_weight', 'total_sales', 'rating', 'outlet_age', 'sales_category',
       'visibility_category', 'rating_category'],
      dtype='str')

In [32]:
df.to_csv(
    "BlinkIT_Cleaned.csv",
    index=False
)

In [34]:
import pandas as pd
from sqlalchemy import create_engine

# 1. Your exact server name from SSMS
SERVER = r"DESKTOP-910SHFC\SUDHEER_JYO"
DATABASE = "BlinkIT_Database"  # <-- Change this to your specific database name

# 2. Build the connection string
connection_string = (
    f"mssql+pyodbc://@{SERVER}/{DATABASE}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
)

# 3. Connect
engine = create_engine(connection_string)
print("Connected successfully!")


Connected successfully!


In [36]:
# Replace "your_new_table_name" with whatever you want to call it in SSMS
table_name = "blinkit_sales"

# Upload the DataFrame to SQL Server
df.to_sql(
    name=table_name,
    con=engine,
    if_exists="replace",
    index=False,
)

print(f"Success! '{table_name}' has been uploaded to SSMS.")


Success! 'blinkit_sales' has been uploaded to SSMS.
